[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module4/Advanced_RNN_Theory.ipynb)

# Advanced RNN Theory
**Dr. Dave Wanik - University of Connecticut**

----------------------------


**Convolution and Pooling (1D - for sequences!)**
Pay attention to how the output shape changes when you include convolution and pooling.

The rows become fewer (due to the convolutions/when the kernel hits the wall) and the columns become your number of 'feature maps'. Convolution does elementwise multiplication and summing, so your original shape of the input features (i.e. 9 stocks) gets destroyed. See below.

For pooling, if you have any remainders - they simply get chopped out. So try to pick a kernel that prevents this, or include some padding (add 0s to help maintain shape and retain all information)

**Recurrent Dropout vs. Dropout**
Recurrent dropout occurs within a SimpleRNN, LSTM or GRU layer. It is DIFFERENT than a DROPOUT layer.

**Bidirectional Layers**
Sequences can be read both forwards and backwards, and so this Bidirectional() layer wraps around SimpleRNN, LSTM or GRU and fits two models at the same time (takes twice as long to run). Yes, they can also be stacked!

**Monster**
Put all of these concepts together and explain the output shape and trainable parameters... good luck!!!

In [1]:
# import modules
# standard modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# RNN-specific modules
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report,accuracy_score
from tensorflow.keras import layers, Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D
from tensorflow.keras.layers import Dense, Dropout, SimpleRNN, GRU, LSTM
from tensorflow.keras.callbacks import EarlyStopping

# reproducibility: same seed every run (numbers on CPU match exactly; a GPU may drift a little)
import keras
keras.utils.set_random_seed(5509)


🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 12 — Conv1D + MaxPooling1D on a sequence, by hand
- A 21 x 9 sample, kernel 2, one filter -> 20 x 1 (rows = 21-2+1, columns = filters). It's Conv1D, NEVER Conv2D, on sequences.
- Params: a 1 x 2 kernel per column (9 columns) + 1 bias = 19.
- MaxPooling1D(2) is the domino: 20 x 1 -> 10 x 1. Into a SimpleRNN(30): 10 spins -> 1 x 30, 960 params, 1,010 total.
- More filters -> richer sequences (20 x 3 -> 10 x 3). Pull input_shape from X_train, never hard-code it.
-->


# Intro to Convolution and Pooling (Sequences)

![alt text](https://qph.fs.quoracdn.net/main-qimg-523434af0d21bb0b59454aa9563cc90b.webp)

In [2]:
# it does not matter how many features you have - the convolution for a single
# feature map will recode your 9 features into 1 via elementwise multiplication and summing

# link: https://www.quora.com/What-does-it-mean-by-1D-convolutional-neural-network

# there are more examples on text sequences, and I'll share these in future videos!

# Single SimpleRNN with Conv1D and MaxPool1D

In [3]:
# same as Excel Spreadsheet
n_features = 9 # 9 stock prices (WMT, GOOG, NETF, GE, AMD...)
n_steps = 21 # lookback 21 days

# my data is 9 columns and 21 days long

# for Conv1D, you can play with the filters and kernel size

# define model
model = Sequential()
# remember - convolution does elementwise multiplication and summation!
model.add(Conv1D(filters=3, # these are the new columns that will be created (YOU GET TO PICK THIS!)
                 kernel_size=2, # this dictates the size of the modified lookback (YOU GET TO PICK THIS!)
                 input_shape=(n_steps,n_features))) # notice how input shape goes in first layer
# you are just downsampling that 1D array
model.add(MaxPooling1D(2)) # the output from pooling is still a sequence (but is richer and denser)
# the new sequence we create will go into a SimpleRNN layer
model.add(SimpleRNN(30, activation='relu'))
# the output from this layer will be (None, 30) and we just return the last hidden state
model.add(Dropout(0.1)) # pick a number between 0.1 and 0.3
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse',metrics=['mae'])
model.summary()

C:\Users\dww05002\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 20, 3)          │            57 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 10, 3)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 30)             │         1,020 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,108 (4.33 KB)

 Trainable params: 1,108 (4.33 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# convolutional layer
# 9 input columns get transformed into 1 output column
# kernel width of 2
# 21 - 2 + 1 = 20 (shape)
# 1 is equal to the number of feature maps

# elementwise mutliplication and summation
# weights + biases
9*2*1 + 1  # 9 = 9 col inputs * 2 = kernel width * 1 output feature maps + 1 (for bias on one feature maps/sequences)

19

In [5]:
# simpleRNN trainable parms
# g × [h(h+i) + h]
# 1 x [30(30+1)+30]
1*(30*(30+1)+30)

960

In [6]:
# convolution layer
# elementwise mutliplication and summation
# weights + biases
9*3*2 + 3 # 9 = 9 col inputs * 2 = kernel width * 3 output feature maps + 3 (for bias on 3 feature maps/sequences)

57

🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 13 — Recurrent dropout, stacking, bidirectional (the monsters)
- Recurrent dropout regularizes the recurrent connection, not just the inputs.
- Stack with return_sequences=True - more layers is NOT automatically better; it's a hyperparameter.
- Bidirectional(LSTM(3)) reads forward and backward with two independent cells and CONCATENATES -> width 6; patterns hard to see forward sometimes pop out backward.
- Build Monster #1 and Monster #2 and read every parameter count off summary().
-->


# Intro to Recurrent Dropout
Imagine some of the weights being turned off in each of the feed-forward network WITHIN the recurrent layer!

![alt text](https://miro.medium.com/max/2250/1*goJVQs-p9kgLODFNyhl9zA.gif)




# Single LSTM layer with Conv1D and MaxPool1D

In [7]:
# same as Excel Spreadsheet
n_features = 9 # 9 stock prices
n_steps = 21 # lookback

# for Conv1D, you can play with the filters and kernel size

# define model
model = Sequential()
# remember - convolution does elementwise multiplication and pooling!
model.add(Conv1D(filters=1,
                 kernel_size=2,
                 input_shape=(n_steps,n_features))) # notice how input shape goes in first layer
model.add(MaxPooling1D(2))
model.add(LSTM(30, activation='relu', recurrent_dropout=0.2))
model.add(Dropout(0.1)) # pick a number between 0.1 and 0.3
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse',metrics=['mae'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_1 (Conv1D)               │ (None, 20, 1)          │            19 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 10, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 30)             │         3,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,890 (15.20 KB)

 Trainable params: 3,890 (15.20 KB)

 Non-trainable params: 0 (0.00 B)

# Multiple LSTM layers with Conv1D and MaxPool1D and Recurrent Dropout
recurrent_dropout=True as an argument in an LSTM layer is different than a Dropout() layer.

In [8]:
# same as Excel Spreadsheet
n_features = 9 # 9 stock prices
n_steps = 21 # lookback

# for Conv1D, you can play with the filters and kernel size

# define model
model = Sequential()
# remember - convolution does elementwise multiplication and pooling!
model.add(Conv1D(filters=1, kernel_size=2, input_shape=(n_steps,n_features))) # notice how input shape goes in first layer
model.add(MaxPooling1D(2))
# return_sequences is only true for the recurrent layers going INTO other recurrent layers
model.add(LSTM(30, activation='relu', recurrent_dropout=0.1, return_sequences=True))
model.add(LSTM(40, activation='relu', recurrent_dropout=0.1, return_sequences=True))
model.add(LSTM(50, activation='relu', recurrent_dropout=0.1, return_sequences=True))
model.add(LSTM(60, activation='relu', recurrent_dropout=0.1))
model.add(Dropout(0.1)) # pick a number between 0.1 and 0.3
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse',metrics=['mae'])
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 20, 1)          │            19 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 10, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 10, 30)         │         3,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 10, 40)         │        11,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 10, 50)         │        18,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 60)             │        26,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 60)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            61 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 60,120 (234.84 KB)

 Trainable params: 60,120 (234.84 KB)

 Non-trainable params: 0 (0.00 B)

# Monster #1
(stacked Conv1D and Pool1D layers, with recurrent_dropout)

In [9]:
# same as Excel Spreadsheet
n_features = 20 # 20 stock prices
n_steps = 50 # lookback

# for Conv1D, you can play with the filters and kernel size

# define model
model = Sequential()
# remember - convolution does elementwise multiplication and pooling!
model.add(Conv1D(filters=64, kernel_size=2, input_shape=(n_steps,n_features))) # notice how input shape goes in first layer
model.add(MaxPooling1D(2))
model.add(Conv1D(filters=128, kernel_size=2)) # no need for input shape!
model.add(MaxPooling1D(2))
model.add(SimpleRNN(64, activation='relu', return_sequences=True))
model.add(SimpleRNN(120, activation='relu',
                    recurrent_dropout=0.1))
model.add(Dropout(0.1)) # pick a number between 0.1 and 0.3
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse',metrics=['mae'])
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_3 (Conv1D)               │ (None, 49, 64)         │         2,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 23, 128)        │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 11, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 11, 64)         │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 120)            │        22,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           121 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,809 (210.19 KB)

 Trainable params: 53,809 (210.19 KB)

 Non-trainable params: 0 (0.00 B)

# Introduction to Bidirectional Layers
See Google Sheets for overview.

# Single Bidirectional LSTM Layer

In [10]:
from keras.layers import Bidirectional

# same as Excel Spreadsheet
n_features = 9 # 9 stock prices
n_steps = 21 # lookback

# for Conv1D, you can play with the filters and kernel size

# define model
model = Sequential()
model.add(Bidirectional(LSTM(30, activation='relu'),
                        input_shape=(n_steps,n_features)))
model.add(Dropout(0.1)) # pick a number between 0.1 and 0.3
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse',metrics=['mae'])
model.summary()

# if you ever get an error like 'the model has not yet been built'
# it means that you have not provided an input shape to your model,
# note how input is an argument to the Bidirectional layer, NOT the LSTM!
# g*(h*(h+i)+h) # for a regular recurrent layer
# 2*g*h(h+i)+h)
# 4*(30*(30+9)+30) for a LSTM with 30 hidden size
# 2*4*(30*(30+9)+30) for a LSTM with 30 hidden size

C:\Users\dww05002\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 60)             │         9,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 60)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            61 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,661 (37.74 KB)

 Trainable params: 9,661 (37.74 KB)

 Non-trainable params: 0 (0.00 B)

# Stacked Bidirectional Layers

In [11]:
from keras.layers import Bidirectional

# same as Excel Spreadsheet
n_features = 9 # 9 stock prices
n_steps = 21 # lookback

# for Conv1D, you can play with the filters and kernel size

# define model
model = Sequential()
model.add(Bidirectional(LSTM(30, activation='relu', return_sequences=True),
                        input_shape=(n_steps,n_features)))
model.add(Bidirectional(LSTM(10, activation='relu')))
model.add(Dropout(0.1)) # pick a number between 0.1 and 0.3
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse',metrics=['mae'])
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_1 (Bidirectional) │ (None, 21, 60)         │         9,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 20)             │         5,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,301 (59.77 KB)

 Trainable params: 15,301 (59.77 KB)

 Non-trainable params: 0 (0.00 B)

# Monster #2
Convolution, pooling, bidirectional etc. All of it!

In [12]:
from keras.layers import Bidirectional

# same as Excel Spreadsheet
n_features = 50 # 50 stock prices
n_steps = 21 # lookback

# for Conv1D, you can play with the filters and kernel size

# define model
model = Sequential()
model.add(Conv1D(filters=20, kernel_size=2, input_shape=(n_steps,n_features))) # notice how input shape goes in first layer
model.add(MaxPooling1D(2))
model.add(Conv1D(filters=10, kernel_size=2,))
model.add(MaxPooling1D(2))
model.add(Bidirectional(LSTM(30, activation='relu', return_sequences=True)))
model.add(Bidirectional(LSTM(60, activation='relu', return_sequences=True)))
model.add(GRU(40, activation='relu', return_sequences=True))
model.add(Bidirectional(LSTM(10, activation='relu')))
model.add(Dropout(0.1)) # pick a number between 0.1 and 0.3
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse',metrics=['mae'])
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_5 (Conv1D)               │ (None, 20, 20)         │         2,020 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 10, 20)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (None, 9, 10)          │           410 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_6 (MaxPooling1D)  │ (None, 4, 10)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 4, 60)          │         9,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 4, 120)         │        58,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 4, 40)          │        19,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 20)             │         4,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 93,891 (366.76 KB)

 Trainable params: 93,891 (366.76 KB)

 Non-trainable params: 0 (0.00 B)